# 🏏 IPL Cricket Analytics — Exploratory Data Analysis

**Database:** `cricket_companion` (MySQL)  
**Tables used:** `ipl_batting_stats`, `ipl_bowling_stats`, `matches`, `teams`, `players`, `scorecards`  
**Analyses:**
1. Dataset overview & null audit
2. Top 10 batsmen by batting average
3. Top 10 bowlers by economy rate
4. Team win rates by venue
5. Run rate trends across IPL seasons
6. Correlation heatmap — batting stats
7. Boundary percentage vs Strike rate scatter
8. Season-wise team dominance heatmap

---

## 0. Setup & DB Connection

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sqlalchemy import create_engine, text
from dotenv import load_dotenv

# -- Load credentials
load_dotenv()
DB_USER = os.getenv('DB_USER', 'root')
DB_PASS = os.getenv('DB_PASSWORD', '')
DB_HOST = os.getenv('DB_HOST', 'localhost')
DB_PORT = int(os.getenv('DB_PORT', 3306))
DB_NAME = os.getenv('DB_NAME', 'cricket_companion')

engine = create_engine(
    f'mysql+mysqlconnector://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}',
    echo=False,
)

def sql(query, **kwargs):
    with engine.connect() as conn:
        return pd.read_sql(text(query), conn, **kwargs)

print('Connected to', DB_NAME)

In [ ]:
# -- Global plot theme
DARK_BG  = '#0f1117'
CARD_BG  = '#1a1d2e'
ACCENT   = '#6c63ff'
ACCENT2  = '#ff6584'
ACCENT3  = '#43e97b'
TEXT     = '#e2e8f0'
GRID     = '#2d3154'

plt.rcParams.update({
    'figure.facecolor': DARK_BG,  'axes.facecolor': CARD_BG,
    'axes.edgecolor': GRID,       'axes.labelcolor': TEXT,
    'xtick.color': TEXT,          'ytick.color': TEXT,
    'text.color': TEXT,           'grid.color': GRID,
    'grid.linewidth': 0.6,        'figure.dpi': 130,
    'font.family': 'sans-serif',  'font.size': 11,
    'axes.titlesize': 14,         'axes.titleweight': 'bold',
    'axes.titlepad': 14,          'legend.facecolor': CARD_BG,
    'legend.edgecolor': GRID,
})

TEAM_COLORS = {
    'Mumbai Indians':              '#004BA0',
    'Chennai Super Kings':         '#F5A623',
    'Royal Challengers Bengaluru': '#EC1C24',
    'Kolkata Knight Riders':       '#3A225D',
    'Delhi Capitals':              '#00489F',
    'Punjab Kings':                '#ED1B24',
    'Rajasthan Royals':            '#EA1A85',
    'Sunrisers Hyderabad':         '#F7A721',
    'Lucknow Super Giants':        '#A72B2A',
    'Gujarat Titans':              '#1C4A8E',
    'Rising Pune Supergiant':      '#660F57',
    'Kochi Tuskers Kerala':        '#5C2D91',
}
print('Plot theme configured')

---
## 1. Dataset Overview & Null Audit

In [ ]:
tables = ['teams','players','series','matches','scorecards',
          'ipl_batting_stats','ipl_bowling_stats']
counts = {}
with engine.connect() as conn:
    for t in tables:
        (n,) = conn.execute(text(f'SELECT COUNT(*) FROM {t}')).fetchone()
        counts[t] = n

count_df = pd.DataFrame.from_dict(counts, orient='index', columns=['row_count'])
count_df.index.name = 'table'
count_df['row_count'] = count_df['row_count'].map('{:,}'.format)
display(count_df.style
    .set_caption('cricket_companion — Table Row Counts')
    .set_properties(**{'text-align': 'right'})
)

In [ ]:
batting = sql('SELECT * FROM ipl_batting_stats')
bowling = sql('SELECT * FROM ipl_bowling_stats')

def null_audit(df):
    nulls = df.isnull().sum()
    pct   = (nulls / len(df) * 100).round(1)
    return pd.DataFrame({'null_count': nulls, 'null_%': pct, 'dtype': df.dtypes}).query('null_count > 0')

print('=== ipl_batting_stats null audit ===')
display(null_audit(batting))
print()
print('=== ipl_bowling_stats null audit ===')
display(null_audit(bowling))
print('\nNOTE: NULL batting_average = batter never dismissed.')
print('      NULL bowling_average = bowler took 0 wickets that season.')

In [ ]:
season_matches = sql("""
    SELECT s.series_name, COUNT(m.id) AS matches_played
    FROM matches m JOIN series s ON m.series_id = s.id
    WHERE s.series_name LIKE 'IPL%'
    GROUP BY s.series_name ORDER BY s.series_name
""")

fig, ax = plt.subplots(figsize=(14, 4))
bars = ax.bar(
    season_matches['series_name'].str.replace('IPL ', '', regex=False),
    season_matches['matches_played'],
    color=ACCENT, alpha=0.85, edgecolor='none', width=0.6
)
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            str(int(bar.get_height())), ha='center', va='bottom', fontsize=9, color=TEXT)
ax.set_title('Matches Played per IPL Season')
ax.set_xlabel('Season')
ax.set_ylabel('Matches')
ax.set_ylim(0, season_matches['matches_played'].max() * 1.15)
ax.grid(axis='y', alpha=0.4)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

---
## 2. Top 10 Batsmen by Batting Average

In [ ]:
top_batsmen = sql("""
    SELECT
        player_name, team,
        SUM(innings) AS total_innings, SUM(runs_scored) AS total_runs,
        SUM(dismissals) AS total_dismissals, SUM(balls_faced) AS total_balls,
        SUM(fours) AS total_fours, SUM(sixes) AS total_sixes,
        MAX(highest_score) AS highest_score, SUM(fifties) AS fifties, SUM(hundreds) AS hundreds,
        ROUND(SUM(runs_scored)/NULLIF(SUM(dismissals),0), 2) AS batting_avg,
        ROUND(SUM(runs_scored)/NULLIF(SUM(balls_faced),0)*100, 2) AS career_sr
    FROM ipl_batting_stats
    GROUP BY player_name, team
    HAVING total_innings >= 20 AND total_dismissals >= 10
    ORDER BY batting_avg DESC LIMIT 10
""")

top_batsmen.index = range(1, len(top_batsmen)+1)
display(top_batsmen[['player_name','team','total_innings','total_runs',
                      'batting_avg','career_sr','highest_score','fifties','hundreds']]
    .rename(columns={'player_name':'Player','team':'Team','total_innings':'Inn',
                     'total_runs':'Runs','batting_avg':'Average','career_sr':'SR',
                     'highest_score':'HS','fifties':'50s','hundreds':'100s'})
    .style
    .set_caption('Top 10 IPL Batsmen — Career Batting Average (min 20 innings, 10 dismissals)')
    .format({'Average': '{:.2f}', 'SR': '{:.2f}'})
    .background_gradient(cmap='YlOrRd', subset=['Average'])
)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Top 10 IPL Batsmen — Career Statistics', fontsize=16, fontweight='bold', color=TEXT)

ps = top_batsmen.sort_values('batting_avg')
colors_bat = [TEAM_COLORS.get(t, ACCENT) for t in ps['team']]

bars = ax1.barh(ps['player_name'], ps['batting_avg'], color=colors_bat, edgecolor='none', height=0.65)
ax1.set_xlabel('Batting Average')
ax1.set_title('Career Batting Average')
ax1.grid(axis='x', alpha=0.35)
for bar, val in zip(bars, ps['batting_avg']):
    ax1.text(val+0.3, bar.get_y()+bar.get_height()/2, f'{val:.1f}', va='center', fontsize=9, color=TEXT)
ax1.set_xlim(0, ps['batting_avg'].max()*1.18)

ax2.scatter(ps['career_sr'], ps['batting_avg'],
            s=ps['total_runs']/10, c=colors_bat, alpha=0.85, edgecolors='white', linewidths=0.7)
for _, row in ps.iterrows():
    ax2.annotate(row['player_name'].split(' ')[-1], (row['career_sr'], row['batting_avg']),
                 textcoords='offset points', xytext=(5, 4), fontsize=8, color=TEXT, alpha=0.9)
ax2.set_xlabel('Career Strike Rate')
ax2.set_ylabel('Batting Average')
ax2.set_title('Average vs Strike Rate\n(bubble size = career runs)')
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

---
## 3. Top 10 Bowlers by Economy Rate

In [ ]:
top_bowlers = sql("""
    SELECT
        player_name, team,
        SUM(innings_bowled) AS total_innings,
        ROUND(SUM(balls_bowled)/6.0,1) AS total_overs,
        SUM(runs_conceded) AS total_runs, SUM(wickets) AS total_wickets,
        SUM(five_wicket_hauls) AS fifers,
        ROUND(SUM(runs_conceded)/NULLIF(SUM(balls_bowled)/6.0,0),2) AS economy,
        ROUND(SUM(runs_conceded)/NULLIF(SUM(wickets),0),2) AS bowl_avg,
        ROUND(SUM(balls_bowled)/NULLIF(SUM(wickets),0),1) AS bowl_sr
    FROM ipl_bowling_stats
    GROUP BY player_name, team
    HAVING total_overs >= 50 AND total_wickets >= 20
    ORDER BY economy ASC LIMIT 10
""")

top_bowlers.index = range(1, len(top_bowlers)+1)
display(top_bowlers[['player_name','team','total_innings','total_overs',
                      'total_wickets','economy','bowl_avg','bowl_sr','fifers']]
    .rename(columns={'player_name':'Bowler','team':'Team','total_innings':'Inn',
                     'total_overs':'Overs','total_wickets':'Wkts','economy':'Economy',
                     'bowl_avg':'Average','bowl_sr':'SR','fifers':'5W'})
    .style
    .set_caption('Top 10 IPL Bowlers — Economy Rate (min 50 overs, 20 wickets)')
    .format({'Economy': '{:.2f}', 'Average': '{:.2f}', 'SR': '{:.1f}'})
    .background_gradient(cmap='RdYlGn_r', subset=['Economy'])
)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Top 10 IPL Bowlers — Economy & Wicket Analysis', fontsize=16, fontweight='bold', color=TEXT)

bs = top_bowlers.sort_values('economy', ascending=False)
colors_bowl = [TEAM_COLORS.get(t, ACCENT2) for t in bs['team']]

bars = ax1.barh(bs['player_name'], bs['economy'], color=colors_bowl, edgecolor='none', height=0.65)
ax1.set_xlabel('Economy Rate (runs/over)')
ax1.set_title('Career Economy Rate (lower = better)')
ax1.grid(axis='x', alpha=0.35)
for bar, val in zip(bars, bs['economy']):
    ax1.text(val+0.05, bar.get_y()+bar.get_height()/2, f'{val:.2f}', va='center', fontsize=9, color=TEXT)
ax1.set_xlim(0, bs['economy'].max()*1.18)

ax2.scatter(bs['economy'], bs['total_wickets'],
            s=bs['total_overs']*2, c=colors_bowl, alpha=0.85, edgecolors='white', linewidths=0.7)
for _, row in bs.iterrows():
    ax2.annotate(row['player_name'].split(' ')[-1], (row['economy'], row['total_wickets']),
                 textcoords='offset points', xytext=(5, 4), fontsize=8, color=TEXT, alpha=0.9)
ax2.set_xlabel('Economy Rate')
ax2.set_ylabel('Total Wickets')
ax2.set_title('Economy vs Total Wickets\n(bubble size = overs bowled)')
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

---
## 4. Team Win Rates by Venue

In [ ]:
win_rate = sql("""
    SELECT t.team_name,
        COUNT(DISTINCT m.id) AS matches_played,
        SUM(CASE WHEN m.winner_id = t.id THEN 1 ELSE 0 END) AS wins,
        ROUND(SUM(CASE WHEN m.winner_id = t.id THEN 1 ELSE 0 END)
              /NULLIF(COUNT(DISTINCT m.id),0)*100, 1) AS win_pct
    FROM teams t
    JOIN matches m ON (m.team1_id = t.id OR m.team2_id = t.id)
    GROUP BY t.id, t.team_name
    HAVING matches_played >= 5
    ORDER BY win_pct DESC
""")

display(win_rate.style
    .set_caption('Team Win Rates — All IPL Seasons')
    .format({'win_pct': '{:.1f}%'})
    .background_gradient(cmap='YlGn', subset=['win_pct'])
)

In [ ]:
fig, ax = plt.subplots(figsize=(13, 6))
wr = win_rate.sort_values('win_pct')
colors = [TEAM_COLORS.get(t, ACCENT) for t in wr['team_name']]

bars = ax.barh(wr['team_name'], wr['win_pct'], color=colors, edgecolor='none', height=0.6)
ax.axvline(50, color='white', linestyle='--', alpha=0.4, linewidth=1, label='50% mark')
for bar, val, mp in zip(bars, wr['win_pct'], wr['matches_played']):
    ax.text(val+0.4, bar.get_y()+bar.get_height()/2,
            f'{val:.1f}%  ({mp} games)', va='center', fontsize=9, color=TEXT)
ax.set_xlabel('Win Percentage')
ax.set_title('IPL Team Win Rates (all seasons)', fontsize=15)
ax.set_xlim(0, 80)
ax.grid(axis='x', alpha=0.3)
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
venue_wins = sql("""
    SELECT m.venue, t.team_name,
        COUNT(*) AS total_matches,
        SUM(CASE WHEN m.winner_id = t.id THEN 1 ELSE 0 END) AS wins
    FROM matches m
    JOIN teams t ON (m.team1_id = t.id OR m.team2_id = t.id)
    WHERE m.venue != 'Unknown'
    GROUP BY m.venue, t.team_name
    HAVING total_matches >= 5
""")
venue_wins['win_pct'] = (venue_wins['wins']/venue_wins['total_matches']*100).round(1)

top_venues = venue_wins.groupby('venue')['total_matches'].sum().nlargest(6).index.tolist()
vw = venue_wins[venue_wins['venue'].isin(top_venues)]

pivot = vw.pivot_table(index='team_name', columns='venue',
                       values='win_pct', aggfunc='mean').fillna(0)
pivot.columns = [c.split(',')[0].strip()[:28] for c in pivot.columns]

fig, ax = plt.subplots(figsize=(15, 7))
sns.heatmap(pivot, annot=True, fmt='.0f', linewidths=0.4,
            cmap='RdYlGn', center=50, vmin=0, vmax=100, ax=ax,
            cbar_kws={'label': 'Win %', 'shrink': 0.7}, annot_kws={'size': 9})
ax.set_title('Team Win % at Top 6 IPL Venues', fontsize=15)
ax.set_xlabel('Venue')
ax.set_ylabel('Team')
plt.xticks(rotation=35, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

---
## 5. Run Rate Trends Across IPL Seasons

In [ ]:
season_bat = sql("""
    SELECT season,
        ROUND(AVG(runs_scored),2)        AS avg_runs,
        ROUND(AVG(strike_rate),2)        AS avg_sr,
        ROUND(AVG(boundary_percentage),2) AS avg_bdry_pct,
        ROUND(AVG(batting_average),2)    AS avg_bat_avg,
        SUM(sixes)                       AS total_sixes,
        SUM(fours)                       AS total_fours,
        COUNT(DISTINCT player_name)      AS unique_batters
    FROM ipl_batting_stats
    WHERE season != 'Unknown'
    GROUP BY season ORDER BY season
""")
season_bat['season'] = season_bat['season'].astype(str)

display(season_bat.style
    .set_caption('IPL Season-wise Batting Trends')
    .format({'avg_runs':'{:.1f}','avg_sr':'{:.1f}','avg_bdry_pct':'{:.1f}','avg_bat_avg':'{:.2f}'})
    .background_gradient(cmap='Blues', subset=['avg_sr'])
)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('IPL Batting Trends Across Seasons', fontsize=17, fontweight='bold', color=TEXT)

x = season_bat['season']

ax = axes[0, 0]
ax.fill_between(x, season_bat['avg_sr'], alpha=0.18, color=ACCENT)
ax.plot(x, season_bat['avg_sr'], marker='o', color=ACCENT, linewidth=2.2,
        markersize=7, markerfacecolor=DARK_BG)
ax.set_title('Avg Strike Rate per Season')
ax.set_ylabel('Strike Rate')
ax.grid(alpha=0.3)
ax.tick_params(axis='x', rotation=45)

ax = axes[0, 1]
ax.fill_between(x, season_bat['avg_bdry_pct'], alpha=0.18, color=ACCENT2)
ax.plot(x, season_bat['avg_bdry_pct'], marker='s', color=ACCENT2, linewidth=2.2,
        markersize=7, markerfacecolor=DARK_BG)
ax.set_title('Avg Boundary % per Season')
ax.set_ylabel('Boundary %')
ax.grid(alpha=0.3)
ax.tick_params(axis='x', rotation=45)

ax = axes[1, 0]
ax.bar(x, season_bat['total_fours'], 0.5, label='Fours', color=ACCENT3, alpha=0.8)
ax.bar(x, season_bat['total_sixes'], 0.5, bottom=season_bat['total_fours'],
       label='Sixes', color=ACCENT2, alpha=0.8)
ax.set_title('Boundaries per Season (Fours + Sixes)')
ax.set_ylabel('Count')
ax.legend()
ax.grid(axis='y', alpha=0.3)
ax.tick_params(axis='x', rotation=45)

ax = axes[1, 1]
ax.fill_between(x, season_bat['avg_bat_avg'].fillna(0), alpha=0.18, color='#f9c74f')
ax.plot(x, season_bat['avg_bat_avg'].fillna(0), marker='^', color='#f9c74f',
        linewidth=2.2, markersize=7, markerfacecolor=DARK_BG)
ax.set_title('Avg Batting Average per Season')
ax.set_ylabel('Batting Average')
ax.grid(alpha=0.3)
ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

---
## 6. Correlation Heatmap — Batting Stats

In [ ]:
bat_numeric = batting[[
    'innings','runs_scored','balls_faced','dismissals',
    'fours','sixes','strike_rate','batting_average',
    'boundary_percentage','highest_score','fifties','hundreds'
]].apply(pd.to_numeric, errors='coerce')
bat_numeric = bat_numeric.dropna(subset=['strike_rate','boundary_percentage'])

corr = bat_numeric.corr()
labels = {
    'innings':'Innings','runs_scored':'Runs','balls_faced':'Balls',
    'dismissals':'Dismissals','fours':'4s','sixes':'6s',
    'strike_rate':'Strike Rate','batting_average':'Bat Avg',
    'boundary_percentage':'Boundary %','highest_score':'High Score',
    'fifties':'50s','hundreds':'100s'
}
corr.index   = [labels.get(c, c) for c in corr.index]
corr.columns = [labels.get(c, c) for c in corr.columns]

mask = np.triu(np.ones_like(corr, dtype=bool))

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f',
            cmap='coolwarm', center=0, vmin=-1, vmax=1,
            linewidths=0.5, ax=ax,
            cbar_kws={'label': 'Pearson r', 'shrink': 0.8},
            annot_kws={'size': 9})
ax.set_title('Batting Stats Correlation Heatmap', fontsize=15)
plt.xticks(rotation=40, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

corr_vals = corr.unstack().reset_index()
corr_vals.columns = ['Stat A','Stat B','r']
corr_vals = corr_vals[corr_vals['Stat A'] != corr_vals['Stat B']].drop_duplicates(subset=['r'])
print('\nTop 5 NEGATIVE correlations:')
display(corr_vals.nsmallest(5,'r')[['Stat A','Stat B','r']].reset_index(drop=True))
print('\nTop 5 POSITIVE correlations:')
display(corr_vals.nlargest(5,'r')[['Stat A','Stat B','r']].reset_index(drop=True))

---
## 7. Player Style Quadrants — Boundary % vs Strike Rate

In [ ]:
career = sql("""
    SELECT player_name, team,
        SUM(innings) AS innings, SUM(runs_scored) AS runs, SUM(balls_faced) AS balls,
        SUM(dismissals) AS dismissals, SUM(fours) AS fours, SUM(sixes) AS sixes,
        ROUND(SUM(runs_scored)/NULLIF(SUM(balls_faced),0)*100,2) AS career_sr,
        ROUND(SUM(runs_scored)/NULLIF(SUM(dismissals),0),2) AS career_avg,
        ROUND((SUM(fours)+SUM(sixes))/NULLIF(SUM(balls_faced),0)*100,2) AS career_bdry_pct
    FROM ipl_batting_stats
    GROUP BY player_name, team
    HAVING innings >= 15 AND balls >= 200
""")
career = career.dropna(subset=['career_sr','career_bdry_pct','career_avg'])

med_sr   = career['career_sr'].median()
med_bdry = career['career_bdry_pct'].median()

def quadrant(row):
    hi_sr   = row['career_sr']      >= med_sr
    hi_bdry = row['career_bdry_pct'] >= med_bdry
    if hi_sr and hi_bdry:       return 'Aggressive Hitter'
    elif hi_sr:                 return 'Running Machine'
    elif hi_bdry:               return 'Boundary Accumulator'
    else:                       return 'Anchor Batsman'

career['style'] = career.apply(quadrant, axis=1)
style_colors = {
    'Aggressive Hitter':     ACCENT2,
    'Running Machine':       ACCENT3,
    'Boundary Accumulator':  '#f9c74f',
    'Anchor Batsman':        ACCENT,
}

fig, ax = plt.subplots(figsize=(14, 9))
for style, group in career.groupby('style'):
    ax.scatter(group['career_bdry_pct'], group['career_sr'],
               s=group['career_avg'].fillna(20)*4, alpha=0.65,
               label=style, color=style_colors[style],
               edgecolors='white', linewidths=0.4)

ax.axvline(med_bdry, color='white', linestyle='--', alpha=0.35, linewidth=1.2)
ax.axhline(med_sr,   color='white', linestyle='--', alpha=0.35, linewidth=1.2)

for _, row in career.nlargest(15, 'runs').iterrows():
    ax.annotate(row['player_name'].split(' ')[-1],
                (row['career_bdry_pct'], row['career_sr']),
                textcoords='offset points', xytext=(5, 3),
                fontsize=7.5, color=TEXT, alpha=0.9)

ax.set_xlabel('Boundary Percentage (%)', fontsize=12)
ax.set_ylabel('Career Strike Rate', fontsize=12)
ax.set_title('Player Style Quadrants — Strike Rate vs Boundary %\n(bubble size = career batting average)', fontsize=14)
ax.legend(title='Batter Style', fontsize=10)
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

print('\nStyle distribution:')
display(career['style'].value_counts().rename('Count').to_frame())

---
## 8. Season-wise Team Dominance Heatmap

In [ ]:
team_season = sql("""
    SELECT bs.season, t.team_name,
        SUM(bs.innings) AS innings, SUM(bs.runs_scored) AS runs,
        ROUND(AVG(bs.strike_rate),1) AS avg_sr
    FROM ipl_batting_stats bs
    JOIN teams t ON t.team_name = bs.team
    WHERE bs.season != 'Unknown'
    GROUP BY bs.season, t.team_name
    ORDER BY bs.season, runs DESC
""")

pivot_runs = team_season.pivot_table(
    index='team_name', columns='season', values='avg_sr', aggfunc='mean').fillna(0)
pivot_runs = pivot_runs[pivot_runs.gt(0).sum(axis=1) >= 4]

fig, ax = plt.subplots(figsize=(16, 7))
sns.heatmap(pivot_runs, annot=True, fmt='.0f', cmap='YlOrRd', linewidths=0.3, ax=ax,
            cbar_kws={'label':'Avg Strike Rate','shrink':0.7}, annot_kws={'size':8})
ax.set_title('Team Avg Strike Rate by Season (darker = faster scoring)', fontsize=14)
ax.set_xlabel('Season')
ax.set_ylabel('Team')
plt.xticks(rotation=45)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
bowl_season = sql("""
    SELECT bs.season, t.team_name,
        ROUND(SUM(bs.runs_conceded)/NULLIF(SUM(bs.balls_bowled)/6.0,0),2) AS economy,
        SUM(bs.wickets) AS wickets
    FROM ipl_bowling_stats bs
    JOIN teams t ON t.team_name = bs.team
    WHERE bs.season != 'Unknown'
    GROUP BY bs.season, t.team_name
    HAVING SUM(bs.balls_bowled) >= 60
""")

pivot_eco = bowl_season.pivot_table(
    index='team_name', columns='season', values='economy', aggfunc='mean')
pivot_eco = pivot_eco[pivot_eco.notna().sum(axis=1) >= 4]

fig, ax = plt.subplots(figsize=(16, 6))
sns.heatmap(pivot_eco, annot=True, fmt='.1f', cmap='RdYlGn_r', linewidths=0.3, ax=ax,
            cbar_kws={'label':'Economy Rate','shrink':0.7}, annot_kws={'size':8})
ax.set_title('Team Bowling Economy by Season (greener = more economical)', fontsize=14)
ax.set_xlabel('Season')
ax.set_ylabel('Team')
plt.xticks(rotation=45)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

---
## 9. Key Insights Summary

In [ ]:
print('=' * 60)
print('  IPL EDA -- KEY INSIGHTS')
print('=' * 60)

best_bat = sql("""
    SELECT player_name, SUM(runs_scored) AS runs,
           ROUND(SUM(runs_scored)/NULLIF(SUM(dismissals),0),2) AS avg
    FROM ipl_batting_stats
    GROUP BY player_name HAVING SUM(innings)>=20
    ORDER BY avg DESC LIMIT 1
""").iloc[0]

best_eco = sql("""
    SELECT player_name,
           ROUND(SUM(runs_conceded)/NULLIF(SUM(balls_bowled)/6.0,0),2) AS eco
    FROM ipl_bowling_stats
    GROUP BY player_name HAVING SUM(balls_bowled)>=300
    ORDER BY eco ASC LIMIT 1
""").iloc[0]

best_team = win_rate.iloc[0]

sr_trend  = season_bat.sort_values('season')
sr_start  = sr_trend.iloc[0]
sr_end    = sr_trend.iloc[-1]

print(f'\n1. Best batting average (>=20 innings): {best_bat["player_name"]} ({best_bat["avg"]:.2f})')
print(f'2. Most economical bowler (>=300 balls): {best_eco["player_name"]} (Eco: {best_eco["eco"]:.2f})')
print(f'3. Best win rate team:  {best_team["team_name"]} ({best_team["win_pct"]:.1f}% from {best_team["matches_played"]} games)')
print(f'4. Strike rate: Season {sr_start["season"]} avg SR = {sr_start["avg_sr"]:.1f}'
      f' --> Season {sr_end["season"]} avg SR = {sr_end["avg_sr"]:.1f}')
print(f'5. Boundary %: {sr_start["avg_bdry_pct"]:.1f}% -> {sr_end["avg_bdry_pct"]:.1f}% over dataset.')
print('\nCorrelation key findings:')
print('  * Runs scored is highly correlated with balls faced.')
print('  * Strike rate and boundary % are positively correlated.')
print('  * Batting average shows weaker SR correlation -- anchors can average high.')
print('=' * 60)